[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/paper_implementation/blob/main/flow_basic.ipynb)

# flow_basic — FashionMNIST 조건부 Flow 실습

FM → Rectified Flow/Reflow → CM → CTM → Shortcut → MeanFlow 순서로 **무엇을 예측하고 어떻게 학습·생성하는지** 비교한다.
모든 생성 모델은 `meanflow_minimal`의 conditional DiT를 기반으로 독립 학습한다. 공통 입력은 32×32 FashionMNIST와 클래스 라벨이며, Muon+AdamW를 사용한다.

각 절은 **핵심 식 → 학습 → 구현 확인 → 클래스별 조건부 생성 → 결과 해석** 순서다. 학습 목표·시간 분포·teacher 사용은 방법별로 유지한다.
CM은 additive noise 기반 standalone CT 실습, CTM은 FM teacher와 soft consistency를 사용하는 적응 구현이다. 원 논문의 대규모 benchmark 재현과 구분한다.

| 절 | 논문 | 원문 |
| --- | --- | --- |
| FM | Flow Matching for Generative Modeling | [2210.02747](https://arxiv.org/abs/2210.02747) |
| RF | Flow Straight and Fast: Learning to Generate and Transfer Data with Rectified Flow | [2209.03003](https://arxiv.org/abs/2209.03003) |
| CM | Consistency Models | [2303.01469](https://arxiv.org/abs/2303.01469) |
| CTM | Consistency Trajectory Models | [2310.02279](https://arxiv.org/abs/2310.02279) |
| Shortcut | One Step Diffusion via Shortcut Models | [2410.12557](https://arxiv.org/abs/2410.12557) |
| MeanFlow | Mean Flows for One-step Generative Modeling | [2505.13447](https://arxiv.org/abs/2505.13447) |
| Flow Map | Generalised Flow Maps for Few-Step Generative Modelling on Riemannian Manifolds | [2510.21608](https://arxiv.org/abs/2510.21608) |

## 0. 공통 설정

`TRAIN_STEPS=20_000`은 학습 상한이다. `EARLY_STOP_AT`은 **업데이트를 마친 뒤 중단할 절대 step**으로, `None`이면 상한까지 학습한다.
모든 모델(FM, RF1, RF2, CM, CTM, Shortcut, MeanFlow)은 기본적으로 **각각 5,000 step**에서 중단한다.
실습 시간을 제한하는 설정이며, 수렴이나 동일한 생성 품질을 보장하는 임계값은 아니다. 모델별 변경은 `EARLY_STOP_AT`에서 지정한다.
일정 step에서 끊는 방식이며 loss plateau 판정은 사용하지 않는다. 특히 MeanFlow adaptive loss 값만으로 수렴을 판정하지 않는다.

학습률은 minimal처럼 상수다. 조기 종료는 학습률이나 CM curriculum의 20,000-step 기준을 다시 계산하지 않는다.
각 모델 학습 중에도 고정 noise·라벨의 PNG를 저장한다. 학습 셀을 중단하면 가능한 경우 마지막 완료 step의 모델과 optimizer를 저장한다.


In [ ]:
# @title 0-1. Install / imports / configuration
%pip -q install datasets scipy pandas matplotlib

import copy
import math
import random
import time
import uuid
from pathlib import Path
from contextlib import contextmanager

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import sqrtm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.func import jvp
from torch.utils.data import DataLoader
from datasets import load_dataset
from torchvision import datasets as tv_datasets, transforms
from torchvision.transforms import ToTensor


# 재현성과 실행 장치
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.mha.set_fastpath_enabled(False)


# 학습 상한과 모델별 조기 종료
TRAIN_STEPS = 20_000
EARLY_STOP_AT = {
    "FM": 5_000,
    "RF1": 5_000,
    "RF2": 5_000,
    "CM": 5_000,
    "CTM": 5_000,
    "Shortcut": 5_000,
    "MeanFlow": 5_000,
}

# 데이터와 공통 DiT 크기
BATCH_SIZE = 128
IMAGE_SIZE = 32
CHANNELS = 1
PATCH_SIZE = 4
MODEL_DIM = 224
MODEL_DEPTH = 4
MODEL_HEADS = 8
FOURIER_DIM = 128

# 조건부 생성 클래스
NUM_CLASSES = 10
CLASS_NAMES = (
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
)

# Muon + AdamW
MUON_LR = 2e-2
ADAMW_LR = 3e-4
MUON_MOMENTUM = 0.95
MUON_BACKEND = "auto"  # auto / native / fallback; actual backend is saved
WEIGHT_DECAY = 0.0
ADAMW_BETAS = (0.9, 0.99)

# CM / CTM의 학습 target
TARGET_EMA_DECAY = 0.999  # CM/CTM training targets only

# MeanFlow의 시간 샘플링과 손실
P_MEAN = -0.4
P_STD = 1.0
DATA_PROPORTION = 0.75
NORM_EPS = 0.01

# 모델별 학습 설정
CM_GRID_START = 32
CM_GRID_END = 1280
RF_TEACHER_NFE = 32
CTM_TEACHER_NFE = 8
CTM_DENOISE_WEIGHT = 1.0
SHORTCUT_LEVELS = 16
SHORTCUT_FM_FRACTION = 0.75

# 로그와 조건부 생성 결과 저장
LOG_EVERY = 50
DIAG_EVERY = 250
SAMPLE_EVERY = 1000
DIAG_BATCH = 64
SAMPLES_PER_CLASS = 4
SAMPLE_BATCH = 32

# 공통 평가
EVAL_N = 1000
NFE_LIST = [1, 2, 4, 8, 16]
DEFAULT_NFE = {
    "FM": 16,
    "RF1": 16,
    "RF2": 16,
    "CM": 1,
    "CTM": 2,
    "Shortcut": 1,
    "MeanFlow": 1,
}
FEATURE_EPOCHS = 3

# 데이터와 실행 결과 경로
DATA_ROOT = (
    Path("/content/fashion_mnist_data")
    if Path("/content").exists()
    else Path("flow_basic_runs/data")
)
OUTPUT_ROOT = (
    Path("/content/flow_basic_runs")
    if Path("/content").exists()
    else Path("flow_basic_runs")
)
RUN_DIR = OUTPUT_ROOT / (time.strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8])
RUN_DIR.mkdir(parents=True, exist_ok=False)


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_all(SEED)
MODELS = {}
LOGS = {}
RUN_INFO = {}
RESULTS = {}
print("device:", DEVICE, "torch:", torch.__version__)
print("run:", RUN_DIR)
print("early stops:", EARLY_STOP_AT)

if DEVICE.type != "cuda":
    print("CPU에서는 작은 실행 검증만 권장합니다. 본 학습은 Colab GPU를 사용하세요.")

In [ ]:
# @title 0-2. FashionMNIST: 28 -> 32 padding, [-1,1], labels
to_tensor = ToTensor()


def transform_image(image):
    image = to_tensor(image)
    padding = (IMAGE_SIZE - image.shape[-1]) // 2

    return F.pad(image, (padding,) * 4, value=0.0) * 2.0 - 1.0


def collate_fashion(batch):
    return (
        torch.stack([transform_image(row["image"]) for row in batch]),
        torch.tensor([row["label"] for row in batch], dtype=torch.long),
    )


try:
    dataset = load_dataset("zalando-datasets/fashion_mnist")
    train_data, test_data = dataset["train"], dataset["test"]
    collate_fn = collate_fashion
    print("FashionMNIST: Hugging Face")
except Exception as error:
    print("Hugging Face download failed; torchvision fallback:", type(error).__name__)
    transform = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Pad((IMAGE_SIZE - 28) // 2),
            transforms.Normalize((0.5,), (0.5,)),
        ]
    )
    train_data = tv_datasets.FashionMNIST(
        str(DATA_ROOT), train=True, download=True, transform=transform
    )
    test_data = tv_datasets.FashionMNIST(
        str(DATA_ROOT), train=False, download=True, transform=transform
    )
    collate_fn = None

loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=0,
    pin_memory=DEVICE.type == "cuda",
    collate_fn=collate_fn,
    generator=loader_generator,
)
test_loader = DataLoader(
    test_data,
    batch_size=DIAG_BATCH,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
)
preview_images, preview_labels = next(iter(test_loader))
assert preview_images.shape[1:] == (CHANNELS, IMAGE_SIZE, IMAGE_SIZE)
assert preview_images.min() >= -1 and preview_images.max() <= 1
fig, axes = plt.subplots(2, 8, figsize=(12, 4))

for ax, image, label in zip(axes.flat, preview_images, preview_labels):
    ax.imshow((image[0] + 1) / 2, cmap="gray", vmin=0, vmax=1)
    ax.set_title(CLASS_NAMES[int(label)], fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

### 공통 conditional DiT

minimal의 별도 시간·구간 임베딩, 클래스 임베딩, adaLN-Zero 초기화, JVP 대응 attention을 사용한다.
`model(z, t, interval, labels)`에서 FM/RF/CM은 `interval=0`, CTM/MeanFlow는 `t-s`, Shortcut은 전진 길이 `d`를 전달한다.
Shortcut은 noise=0 → data=1, 나머지 선형 Flow는 data=0 → noise=1 방향이다.
같은 구조를 공유하되 모델 파라미터는 각각 새로 만든다. 클래스 조건은 모든 teacher·target·sampling 호출에 유지한다.


In [ ]:
class ScalarEmbed(nn.Module):
    def __init__(self, dim, fourier_dim=FOURIER_DIM):
        super().__init__()
        self.fourier_dim = fourier_dim
        self.mlp = nn.Sequential(
            nn.Linear(fourier_dim, dim),
            nn.SiLU(),
            nn.Linear(dim, dim),
        )

    def forward(self, scalar):
        half = self.fourier_dim // 2
        frequencies = torch.exp(
            -math.log(10000.0)
            * torch.arange(
                half,
                device=scalar.device,
                dtype=scalar.dtype,
            )
            / half
        )
        phase = scalar[:, None] * frequencies[None, :] * 2.0 * math.pi
        embedding = torch.cat(
            [phase.cos(), phase.sin()],
            dim=-1,
        )

        return self.mlp(embedding)


class Block(nn.Module):
    def __init__(self, dim=MODEL_DIM, heads=MODEL_HEADS):
        super().__init__()
        self.norm1 = nn.LayerNorm(
            dim,
            elementwise_affine=False,
            eps=1e-6,
        )
        self.norm2 = nn.LayerNorm(
            dim,
            elementwise_affine=False,
            eps=1e-6,
        )
        self.attn = nn.MultiheadAttention(
            dim,
            heads,
            batch_first=True,
        )
        self.mlp = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(approximate="tanh"),
            nn.Linear(4 * dim, dim),
        )
        self.mod = nn.Sequential(
            nn.SiLU(),
            nn.Linear(dim, 6 * dim),
        )
        nn.init.zeros_(self.mod[-1].weight)
        nn.init.zeros_(self.mod[-1].bias)

    def forward(self, tokens, conditioning):
        (
            shift1,
            scale1,
            gate1,
            shift2,
            scale2,
            gate2,
        ) = self.mod(conditioning).chunk(6, dim=-1)

        hidden = self.norm1(tokens)
        hidden = hidden * (1.0 + scale1[:, None, :]) + shift1[:, None, :]
        attended, _ = self.attn(
            hidden,
            hidden,
            hidden,
            need_weights=True,
        )
        tokens = tokens + gate1[:, None, :] * attended

        hidden = self.norm2(tokens)
        hidden = hidden * (1.0 + scale2[:, None, :]) + shift2[:, None, :]

        return tokens + gate2[:, None, :] * self.mlp(hidden)


class CondDiT(nn.Module):
    def __init__(self, dim=MODEL_DIM, depth=MODEL_DEPTH, patch=PATCH_SIZE):
        super().__init__()
        self.patch = patch
        self.grid = IMAGE_SIZE // patch
        self.pe = nn.Conv2d(
            CHANNELS,
            dim,
            patch,
            patch,
        )
        self.pos = nn.Parameter(torch.zeros(1, self.grid**2, dim))
        self.te = ScalarEmbed(dim)
        self.he = ScalarEmbed(dim)
        self.ye = nn.Embedding(NUM_CLASSES, dim)
        self.blocks = nn.ModuleList([Block(dim, MODEL_HEADS) for _ in range(depth)])
        self.norm = nn.LayerNorm(
            dim,
            elementwise_affine=False,
            eps=1e-6,
        )
        self.fmod = nn.Sequential(
            nn.SiLU(),
            nn.Linear(dim, 2 * dim),
        )
        self.out = nn.Linear(
            dim,
            patch * patch * CHANNELS,
        )
        nn.init.normal_(self.pos, std=0.02)
        nn.init.normal_(self.ye.weight, std=0.02)
        nn.init.zeros_(self.fmod[-1].weight)
        nn.init.zeros_(self.fmod[-1].bias)
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def forward(self, images, t, interval, labels):
        batch_size = images.shape[0]

        tokens = self.pe(images)
        tokens = tokens.flatten(2).transpose(1, 2)
        tokens = tokens + self.pos

        conditioning = self.te(t) + self.he(interval) + self.ye(labels)

        for block in self.blocks:
            tokens = block(tokens, conditioning)

        shift, scale = self.fmod(conditioning).chunk(2, dim=-1)
        tokens = self.norm(tokens)
        tokens = tokens * (1.0 + scale[:, None, :]) + shift[:, None, :]
        patches = self.out(tokens)
        patches = patches.view(
            batch_size,
            self.grid,
            self.grid,
            self.patch,
            self.patch,
            CHANNELS,
        )
        images_out = torch.einsum(
            "nhwpqc->nchpwq",
            patches,
        )

        return images_out.reshape(
            batch_size,
            CHANNELS,
            IMAGE_SIZE,
            IMAGE_SIZE,
        )


def fresh_model():
    seed_all(SEED)

    return CondDiT().to(DEVICE)

In [ ]:
@torch.no_grad()
def ns5(gradient, steps=5, eps=1e-7):
    matrix = gradient.float()
    transpose_back = matrix.shape[0] > matrix.shape[1]

    if transpose_back:
        matrix = matrix.T

    matrix = matrix / (matrix.norm() + eps)

    a = 3.4445
    b = -4.775
    c = 2.0315

    for _ in range(steps):
        gram = matrix @ matrix.T
        matrix = a * matrix + (b * gram + c * gram @ gram) @ matrix

    if transpose_back:
        matrix = matrix.T

    return matrix.to(gradient.dtype)


class MuonFallback(torch.optim.Optimizer):
    def __init__(self, params, lr=0.02, momentum=0.95):
        super().__init__(
            params,
            {
                "lr": lr,
                "momentum": momentum,
            },
        )

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for parameter in group["params"]:
                if parameter.grad is None:
                    continue

                state = self.state[parameter]
                buffer = state.setdefault(
                    "momentum_buffer",
                    torch.zeros_like(parameter.grad),
                )
                buffer.mul_(group["momentum"]).add_(parameter.grad)

                update = ns5(parameter.grad + group["momentum"] * buffer)
                update *= math.sqrt(
                    max(
                        1.0,
                        parameter.shape[0] / parameter.shape[1],
                    )
                )
                parameter.add_(
                    update,
                    alpha=-group["lr"],
                )


def build_optimizers(model):
    muon_params, adam_params, muon_names = [], [], []

    for name, parameter in model.named_parameters():
        if parameter.ndim == 2 and (".attn." in name or ".mlp." in name):
            muon_params.append(parameter)
            muon_names.append(name)
        else:
            adam_params.append(parameter)

    assert muon_params and adam_params
    assert len({id(p) for p in muon_params + adam_params}) == len(
        list(model.parameters())
    )
    native = hasattr(torch.optim, "Muon")

    if MUON_BACKEND not in ("auto", "native", "fallback"):
        raise ValueError("Unknown Muon backend")

    if MUON_BACKEND == "native" and not native:
        raise RuntimeError("This PyTorch runtime has no torch.optim.Muon")

    if native and MUON_BACKEND != "fallback":
        muon = torch.optim.Muon(
            muon_params, lr=MUON_LR, momentum=MUON_MOMENTUM, weight_decay=WEIGHT_DECAY
        )
        backend = "torch.optim.Muon"
    else:
        if WEIGHT_DECAY != 0:
            raise ValueError("minimal MuonFallback supports weight_decay=0 only")

        muon = MuonFallback(muon_params, lr=MUON_LR, momentum=MUON_MOMENTUM)
        backend = "minimal MuonFallback"

    adam = torch.optim.AdamW(
        adam_params, lr=ADAMW_LR, betas=ADAMW_BETAS, eps=1e-8, weight_decay=WEIGHT_DECAY
    )

    return (muon, adam), {
        "backend": backend,
        "muon_names": muon_names,
        "muon_tensors": len(muon_params),
        "adamw_tensors": len(adam_params),
    }

### 학습·생성 공통 처리


In [ ]:
# @title 공통 함수 — 시간·난수·클래스별 생성
def expand_time(t):
    return t[:, None, None, None]


def infinite(loader):
    while True:
        yield from loader


def freeze_copy(model):
    target = copy.deepcopy(model).eval()
    target.requires_grad_(False)

    return target


@torch.no_grad()
def update_target(target, model):
    for target_p, online_p in zip(target.parameters(), model.parameters()):
        target_p.lerp_(online_p, 1.0 - TARGET_EMA_DECAY)


@contextmanager
def isolated_rng(seed=SEED + 10_000):
    python_state, numpy_state = random.getstate(), np.random.get_state()
    devices = [torch.cuda.current_device()] if DEVICE.type == "cuda" else []

    try:
        with torch.random.fork_rng(devices=devices):
            seed_all(seed)
            yield
    finally:
        random.setstate(python_state)
        np.random.set_state(numpy_state)


def balanced_labels(n):
    if n % NUM_CLASSES:
        raise ValueError("Use a sample count divisible by NUM_CLASSES")

    return torch.arange(NUM_CLASSES, device=DEVICE).repeat_interleave(n // NUM_CLASSES)


def make_noise(n, seed):
    generator = torch.Generator(device=DEVICE).manual_seed(seed)

    return torch.randn(
        n, CHANNELS, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE, generator=generator
    )


@torch.no_grad()
def generate_conditioned(model, sampler, labels, steps, seed=SEED + 10_000):
    labels = torch.as_tensor(labels, dtype=torch.long, device=DEVICE)

    if (
        labels.ndim != 1
        or len(labels) == 0
        or ((labels < 0) | (labels >= NUM_CLASSES)).any()
    ):
        raise ValueError("labels must be a nonempty 1D sequence of class IDs")

    noise = make_noise(len(labels), seed)
    was_training = model.training
    model.eval()

    try:
        outputs = []

        with isolated_rng(seed + 1):
            for start in range(0, len(labels), SAMPLE_BATCH):
                outputs.append(
                    sampler(
                        model,
                        noise[start : start + SAMPLE_BATCH],
                        labels[start : start + SAMPLE_BATCH],
                        steps,
                    ).cpu()
                )

        return torch.cat(outputs)
    finally:
        model.train(was_training)


def show_conditional(model, sampler, name, steps, step=None, save=True):
    labels = balanced_labels(NUM_CLASSES * SAMPLES_PER_CLASS)
    fake = generate_conditioned(model, sampler, labels, steps)
    fig, axes = plt.subplots(
        NUM_CLASSES,
        SAMPLES_PER_CLASS,
        figsize=(SAMPLES_PER_CLASS * 1.8, NUM_CLASSES * 1.35),
        squeeze=False,
    )

    for index, ax in enumerate(axes.flat):
        label = int(labels[index])
        ax.imshow(((fake[index, 0] + 1) / 2).clamp(0, 1), cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"y={label} {CLASS_NAMES[label]}", fontsize=8)
        ax.axis("off")

    caption = f"{name} | conditional | NFE={steps}"

    if step is not None:
        caption += f" | train step={step}"

    fig.suptitle(caption)
    fig.tight_layout()

    if save:
        folder = RUN_DIR / name
        folder.mkdir(parents=True, exist_ok=True)
        suffix = f"{step:05d}" if step is not None else "final"
        fig.savefig(folder / f"conditional_{suffix}_nfe{steps}.png", dpi=140)

    plt.show()
    plt.close(fig)

    return fake


def should_stop(name, step):
    stop_at = EARLY_STOP_AT.get(name)

    return stop_at is not None and step >= stop_at

In [ ]:
# @title 체크포인트 — 설정과 학습 상태 저장


def checkpoint_config():
    names = (
        "SEED",
        "TRAIN_STEPS",
        "EARLY_STOP_AT",
        "BATCH_SIZE",
        "IMAGE_SIZE",
        "CHANNELS",
        "PATCH_SIZE",
        "MODEL_DIM",
        "MODEL_DEPTH",
        "MODEL_HEADS",
        "FOURIER_DIM",
        "NUM_CLASSES",
        "MUON_LR",
        "ADAMW_LR",
        "MUON_MOMENTUM",
        "MUON_BACKEND",
        "WEIGHT_DECAY",
        "ADAMW_BETAS",
        "TARGET_EMA_DECAY",
        "P_MEAN",
        "P_STD",
        "DATA_PROPORTION",
        "NORM_EPS",
        "CM_GRID_START",
        "CM_GRID_END",
        "RF_TEACHER_NFE",
        "CTM_TEACHER_NFE",
        "CTM_DENOISE_WEIGHT",
        "SHORTCUT_LEVELS",
        "SHORTCUT_FM_FRACTION",
    )
    config = {name: copy.deepcopy(globals()[name]) for name in names}

    for name in ("CM_SIGMA_MIN", "CM_SIGMA_MAX", "CM_SIGMA_DATA", "CM_RHO"):
        if name in globals():
            config[name] = globals()[name]

    return config


def save_checkpoint(name, model, optimizers, target, step, reason, optimizer_info):
    folder = RUN_DIR / name
    folder.mkdir(parents=True, exist_ok=True)
    state = {
        "method": name,
        "step": step,
        "stop_reason": reason,
        "model": model.state_dict(),
        "optimizers": [opt.state_dict() for opt in optimizers],
        "target": target.state_dict() if target is not None else None,
        "config": checkpoint_config(),
        "optimizer_info": optimizer_info,
        "torch_version": str(torch.__version__),
        "torch_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all() if DEVICE.type == "cuda" else [],
        "loader_rng": loader_generator.get_state(),
    }
    temporary = folder / "last.tmp"
    torch.save(state, temporary)
    temporary.replace(folder / "last.pt")

In [ ]:
# @title 공통 학습 루프 — 업데이트·진단·조기 종료


def train_method(name, model, loss_fn, sampler, target=None, diagnostic=None):
    stop_at = EARLY_STOP_AT.get(name)

    if stop_at is not None and (
        isinstance(stop_at, bool)
        or not isinstance(stop_at, int)
        or not 1 <= stop_at <= TRAIN_STEPS
    ):
        raise ValueError(
            f"{name}: early stop must be None or an integer in [1, TRAIN_STEPS]"
        )

    optimizers, optimizer_info = build_optimizers(model)
    seed_all(SEED)
    loader_generator.manual_seed(SEED)
    iterator = infinite(train_loader)
    logs, started, completed, reason = [], time.perf_counter(), 0, "max_steps"
    model.train()
    print(name, optimizer_info["backend"], "max=", TRAIN_STEPS, "stop_at=", stop_at)

    try:
        for step in range(1, TRAIN_STEPS + 1):
            images, labels = next(iterator)
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            for optimizer in optimizers:
                optimizer.zero_grad(set_to_none=True)
            # Match minimal: FP32 (TF32 allowed), no autocast or gradient clipping.

            loss, auxiliary = loss_fn(model, images, labels, step)

            if not torch.isfinite(loss):
                raise FloatingPointError(f"{name}: nonfinite loss at step {step}")

            loss.backward()
            norm_sq = torch.zeros((), device=DEVICE)

            for parameter in model.parameters():
                if parameter.grad is not None:
                    norm_sq += parameter.grad.detach().float().square().sum()

            if not torch.isfinite(norm_sq):
                raise FloatingPointError(
                    f"{name}: nonfinite gradient norm at step {step}"
                )

            for optimizer in optimizers:
                optimizer.step()

            if target is not None:
                update_target(target, model)

            completed = step
            final_step = should_stop(name, step) or step == TRAIN_STEPS
            log_now = step == 1 or step % LOG_EVERY == 0 or final_step

            if log_now:
                row = {
                    "step": step,
                    "loss": float(loss.detach()),
                    "grad_norm": float(norm_sq.sqrt()),
                }
                row.update({key: float(value) for key, value in auxiliary.items()})

                if diagnostic is not None and (step % DIAG_EVERY == 0 or final_step):
                    with isolated_rng():
                        row.update(diagnostic(model))

                logs.append(row)
                print(name, row)

            if step % SAMPLE_EVERY == 0 and not final_step:
                save_checkpoint(
                    name, model, optimizers, target, step, "running", optimizer_info
                )
                show_conditional(model, sampler, name, DEFAULT_NFE[name], step=step)

            if should_stop(name, step):
                reason = "early_stop_at"
                break
    except KeyboardInterrupt:
        reason = "interrupted"
        print(
            f"{name}: interrupted; saving current state after {completed} completed steps"
        )
    except Exception:
        save_checkpoint(
            name, model, optimizers, target, completed, "failed", optimizer_info
        )
        raise

    elapsed = time.perf_counter() - started
    save_checkpoint(name, model, optimizers, target, completed, reason, optimizer_info)
    frame = pd.DataFrame(logs)
    frame.to_csv(RUN_DIR / name / "training.csv", index=False)
    MODELS[name] = model.eval()
    LOGS[name] = frame
    RUN_INFO[name] = {
        "train_steps": completed,
        "train_s": elapsed,
        "stop_reason": reason,
        "optimizer_backend": optimizer_info["backend"],
    }
    print(name, RUN_INFO[name], "checkpoint:", RUN_DIR / name / "last.pt")

    return model


def show_training(name):
    frame = LOGS[name]
    display(frame.tail())

    if not frame.empty:
        columns = [
            c for c in ("loss", "raw_mse", "boundary_mse", "interval_mse") if c in frame
        ]
        frame.plot(x="step", y=columns, subplots=True, figsize=(8, 2.5 * len(columns)))
        plt.show()

In [ ]:
# @title 0-4. Data-independent preflight: shape / independent conditions / JVP
def verify_common_model():
    model = fresh_model()
    z = make_noise(2, SEED)
    t = torch.full((2,), 0.7, device=DEVICE)
    r = torch.full((2,), 0.2, device=DEVICE)
    labels = torch.arange(2, device=DEVICE) % NUM_CLASSES
    assert model(z, t, t - r, labels).shape == z.shape
    # The zero output head intentionally makes all initial predictions zero.
    # Inspect conditioning itself instead of expecting different initial images.
    c1 = model.te(t) + model.he(r)
    c2 = model.te(r) + model.he(t)
    assert not torch.allclose(c1, c2)
    velocity = torch.randn_like(z)
    out, derivative = jvp(
        lambda zz, tt, rr: model(zz, tt, tt - rr, labels),
        (z, t, r),
        (velocity, torch.ones_like(t), torch.zeros_like(r)),
    )
    assert torch.isfinite(derivative).all()
    (
        out - (velocity - expand_time(t - r) * derivative).detach()
    ).square().mean().backward()
    assert (
        model.out.weight.grad is not None
        and torch.isfinite(model.out.weight.grad).all()
    )
    print("Common DiT forward / role separation / JVP backward: PASS")
    print("parameters:", sum(p.numel() for p in model.parameters()))


verify_common_model()

### 공통 생성 평가

평가용 FashionMNIST 분류기는 중앙 28×28 영역을 사용한다. 생성 이미지에도 동일한 crop과 `[-1,1]` clipping을 적용한다.
feature-FID/KID는 이 분류기 특징에서 계산하는 실습 지표다. 원 논문의 Inception FID와 숫자를 비교하지 않는다.
클래스별 동일 개수의 실제·생성 샘플을 사용하고, 요청 라벨과 분류 예측의 일치율도 기록한다. 분류기 자체의 test accuracy를 먼저 확인한다.
최종 평가 데이터는 조기 종료 판단에 사용하지 않는다. 모든 방법에서 고정 noise·라벨·샘플 개수·NFE를 유지한다.


In [ ]:
# @title 0-5. Feature classifier and balanced evaluation set
class FashionFeatures(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(CHANNELS, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc = nn.Linear(64 * 7 * 7, 128)
        self.cls = nn.Linear(128, NUM_CLASSES)

    def forward(self, images, features=False):
        margin = (images.shape[-1] - 28) // 2
        images = images[..., margin : margin + 28, margin : margin + 28].clamp(-1, 1)
        hidden = F.relu(self.fc(self.conv(images).flatten(1)))

        return hidden if features else self.cls(hidden)


seed_all(SEED)
featnet = FashionFeatures().to(DEVICE)
feature_optimizer = torch.optim.Adam(featnet.parameters(), lr=1e-3)

for epoch in range(FEATURE_EPOCHS):
    featnet.train()

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        feature_optimizer.zero_grad(set_to_none=True)
        loss = F.cross_entropy(featnet(images), labels)
        loss.backward()
        feature_optimizer.step()

    print("feature classifier epoch", epoch + 1, "loss", float(loss.detach()))

featnet.eval().requires_grad_(False)


@torch.no_grad()
def collect_evaluation_data():
    if EVAL_N % NUM_CLASSES:
        raise ValueError("EVAL_N must be divisible by NUM_CLASSES")

    per_class = EVAL_N // NUM_CLASSES
    buckets = [[] for _ in range(NUM_CLASSES)]
    correct, total = 0, 0

    for images, labels in test_loader:
        predictions = featnet(images.to(DEVICE)).argmax(1).cpu()
        correct += (predictions == labels).sum().item()
        total += len(labels)

        for image, label in zip(images, labels):
            bucket = buckets[int(label)]

            if len(bucket) < per_class:
                bucket.append(image)

    assert all(len(bucket) == per_class for bucket in buckets)

    return torch.stack(
        [image for bucket in buckets for image in bucket]
    ), correct / total


real_eval, classifier_accuracy = collect_evaluation_data()
print("feature classifier test accuracy:", classifier_accuracy)


@torch.no_grad()
def extract_features(images):
    return (
        torch.cat(
            [
                featnet(batch.to(DEVICE), features=True).cpu()
                for batch in images.split(SAMPLE_BATCH)
            ]
        )
        .numpy()
        .astype(np.float64)
    )


real_features = extract_features(real_eval)


def fid_np(real, fake):
    delta = real.mean(0) - fake.mean(0)
    real_cov, fake_cov = np.cov(real, rowvar=False), np.cov(fake, rowvar=False)
    root = sqrtm(real_cov @ fake_cov)

    if np.iscomplexobj(root):
        root = root.real

    return float(delta @ delta + np.trace(real_cov + fake_cov - 2 * root))


def kid_np(real, fake):
    n, m, dim = len(real), len(fake), real.shape[1]
    aa, bb, ab = (
        (real @ real.T / dim + 1) ** 3,
        (fake @ fake.T / dim + 1) ** 3,
        (real @ fake.T / dim + 1) ** 3,
    )

    return float(
        (aa.sum() - np.trace(aa)) / (n * (n - 1))
        + (bb.sum() - np.trace(bb)) / (m * (m - 1))
        - 2 * ab.mean()
    )


@torch.no_grad()
def quality_metrics(fake, labels):
    fake = fake.cpu().clamp(-1, 1)
    features = extract_features(fake)
    predictions = torch.cat(
        [
            featnet(batch.to(DEVICE)).argmax(1).cpu()
            for batch in fake.split(SAMPLE_BATCH)
        ]
    )
    histogram = predictions.bincount(minlength=NUM_CLASSES).float() / len(predictions)

    return {
        "feature_fid": fid_np(real_features, features),
        "kid": kid_np(real_features, features),
        "condition_accuracy": float((predictions == labels.cpu()).float().mean()),
        "class_entropy": float(-(histogram * histogram.clamp_min(1e-12).log()).sum()),
    }


def evaluate_method(name, sampler, steps=None):
    steps = DEFAULT_NFE[name] if steps is None else steps
    labels = balanced_labels(EVAL_N)
    started = time.perf_counter()
    fake = generate_conditioned(MODELS[name], sampler, labels, steps)
    metrics = quality_metrics(fake, labels)
    metrics.update(RUN_INFO[name])
    metrics.update(nfe=steps, eval_s=time.perf_counter() - started)
    RESULTS[name] = metrics
    print(name, metrics)

    return fake

## 1. Flow Matching — 순간 속도

\(z_t=(1-t)x+t\epsilon\), \(v_t=\epsilon-x\)로 구성하고 \(v_\theta(z_t,t,y)\)를 MSE로 회귀한다.
조건부 생성은 고정 클래스 \(y\)에서 noise \(t=1\)부터 data \(t=0\)까지 ODE를 적분한다.
이 절의 conditional은 클래스 조건이다. 논문의 conditional probability path에서 말하는 conditional과 구분한다.
**적응:** 선형 path, 소형 conditional DiT, FashionMNIST, Muon+AdamW.


In [ ]:
# @title FM — objective / sampler
def fm_loss(model, images, labels, step):
    t = torch.rand(len(images), device=DEVICE)
    noise = torch.randn_like(images)
    z = (1 - expand_time(t)) * images + expand_time(t) * noise
    prediction = model(z, t, torch.zeros_like(t), labels)
    loss = F.mse_loss(prediction, noise - images)

    return loss, {"raw_mse": loss.detach()}


@torch.no_grad()
def integrate_fm(model, z, t, s, labels, steps, return_traj=False):
    if steps < 1:
        raise ValueError("steps must be positive")

    trajectory = [z.cpu()] if return_traj else None
    dt = (s - t) / steps

    for k in range(steps):
        tau = t + (s - t) * (k / steps)
        z = z + expand_time(dt) * model(z, tau, torch.zeros_like(tau), labels)

        if return_traj:
            trajectory.append(z.cpu())

    return (z, torch.stack(trajectory)) if return_traj else z


@torch.no_grad()
def sample_fm(model, noise, labels, steps):
    t = torch.ones(len(noise), device=noise.device)

    return integrate_fm(model, noise, t, torch.zeros_like(t), labels, steps)

In [ ]:
fm = train_method("FM", fresh_model(), fm_loss, sample_fm)
show_training("FM")

In [ ]:
# @title FM — update verification and class-conditioned generation
z, y = make_noise(NUM_CLASSES, SEED), torch.arange(NUM_CLASSES, device=DEVICE)
t = torch.ones(NUM_CLASSES, device=DEVICE)

with torch.no_grad():
    assert torch.allclose(sample_fm(fm, z, y, 1), z - fm(z, t, torch.zeros_like(t), y))

print("FM Euler sign / one-step update: PASS")
show_conditional(fm, sample_fm, "FM", DEFAULT_NFE["FM"], RUN_INFO["FM"]["train_steps"])
fm_samples = evaluate_method("FM", sample_fm)

In [ ]:
# @title FM — 원하는 클래스 직접 생성 (라벨·seed·NFE 수정)
# 0 T-shirt, 1 Trouser, 2 Pullover, 3 Dress, 4 Coat,
# 5 Sandal, 6 Shirt, 7 Sneaker, 8 Bag, 9 Ankle boot
requested_labels = [0, 0, 1, 1, 7, 7, 9, 9]
requested_nfe = DEFAULT_NFE["FM"]
requested_seed = SEED + 20_000
custom_images = generate_conditioned(
    MODELS["FM"], sample_fm, requested_labels, requested_nfe, seed=requested_seed
)
fig, axes = plt.subplots(
    1, len(requested_labels), figsize=(2 * len(requested_labels), 2.5), squeeze=False
)

for ax, image, label in zip(axes.flat, custom_images, requested_labels):
    ax.imshow(((image[0] + 1) / 2).clamp(0, 1), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"y={label} {CLASS_NAMES[label]}", fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 2. Rectified Flow + Reflow — coupling 바꾸기

1-RF의 선형 회귀식은 앞의 FM과 같다. 동일 초기화·seed·설정으로 학습하면 두 결과가 같아지는 것이 자연스럽다.
2-RF는 고정한 1-RF에서 \(\epsilon \mapsto \hat x\)를 생성하고 \((\epsilon,\hat x,y)\)를 유지해 새 직선 경로를 학습한다.
synthetic endpoint는 배치마다 생성한다. 재학습 비용에는 teacher 적분이 포함된다. 생성 시에도 같은 클래스 조건을 유지한다.


In [ ]:
# @title RF — independent 1-RF, then conditional reflow
rf1 = train_method("RF1", fresh_model(), fm_loss, sample_fm)


def reflow_loss(model, images, labels, step):
    noise = torch.randn_like(images)

    with torch.no_grad():
        endpoint = sample_fm(rf1, noise, labels, RF_TEACHER_NFE)

    t = torch.rand(len(images), device=DEVICE)
    z = (1 - expand_time(t)) * endpoint + expand_time(t) * noise
    prediction = model(z, t, torch.zeros_like(t), labels)
    loss = F.mse_loss(prediction, noise - endpoint)

    return loss, {"raw_mse": loss.detach()}


rf2 = train_method("RF2", fresh_model(), reflow_loss, sample_fm)
show_training("RF1")
show_training("RF2")

In [ ]:
# @title RF — conditional generation before/after reflow
for name in ("RF1", "RF2"):
    show_conditional(
        MODELS[name], sample_fm, name, DEFAULT_NFE[name], RUN_INFO[name]["train_steps"]
    )
    evaluate_method(name, sample_fm)

In [ ]:
# @title RF2 — 원하는 클래스 직접 생성 (라벨·seed·NFE 수정)
# 0 T-shirt, 1 Trouser, 2 Pullover, 3 Dress, 4 Coat,
# 5 Sandal, 6 Shirt, 7 Sneaker, 8 Bag, 9 Ankle boot
requested_labels = [0, 0, 1, 1, 7, 7, 9, 9]
requested_nfe = DEFAULT_NFE["RF2"]
requested_seed = SEED + 20_000
custom_images = generate_conditioned(
    MODELS["RF2"], sample_fm, requested_labels, requested_nfe, seed=requested_seed
)
fig, axes = plt.subplots(
    1, len(requested_labels), figsize=(2 * len(requested_labels), 2.5), squeeze=False
)

for ax, image, label in zip(axes.flat, custom_images, requested_labels):
    ax.imshow(((image[0] + 1) / 2).clamp(0, 1), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"y={label} {CLASS_NAMES[label]}", fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 3. Consistency Models — endpoint consistency

standalone Consistency Training을 실습한다. 인접 noise level의 두 입력에서 online prediction과 stop-gradient EMA target을 일치시킨다.
원 논문의 additive corruption \(z_\sigma=x+\sigma\epsilon\)과 \(\sigma_\min\)에서 identity인 skip/output preconditioning을 사용한다.
같은 \(x,\epsilon\)에서 만든 인접 입력은 CT의 stochastic 근사 구성이다. 유한 간격에서 동일 marginal PF-ODE trajectory 위의 정확한 두 점이라는 뜻은 아니다.

이 절의 noise scale은 다른 Flow의 선형 \(t\)와 다르다. DiT에는 정규화한 log-noise level과 `interval=0`을 전달한다.
**적응:** MSE, 고정 target EMA decay, 소형 DiT/Muon. 격자는 학습 상한을 기준으로 증가한다. 원 benchmark 전체 recipe 재현은 아니다.


In [ ]:
# @title CM — noise curriculum / standalone CT / boundary-preserving sampler
CM_SIGMA_MIN = 0.002
CM_SIGMA_MAX = 80.0
CM_SIGMA_DATA = 0.5
CM_RHO = 7.0


def cm_grid_size(step):
    stages = math.ceil(math.log2(CM_GRID_END / CM_GRID_START))
    stage = int(stages * (step - 1) / max(1, TRAIN_STEPS - 1))

    return min(CM_GRID_END, CM_GRID_START * 2**stage)


def cm_noise_levels(fraction):
    lo, hi = CM_SIGMA_MIN ** (1 / CM_RHO), CM_SIGMA_MAX ** (1 / CM_RHO)

    return (lo + fraction * (hi - lo)) ** CM_RHO


def cm_f(model, z, sigma, labels):
    sigma_image = expand_time(sigma)
    offset = sigma_image - CM_SIGMA_MIN
    cskip = CM_SIGMA_DATA**2 / (offset.square() + CM_SIGMA_DATA**2)
    cout = CM_SIGMA_DATA * offset / (sigma_image.square() + CM_SIGMA_DATA**2).sqrt()
    cin = (sigma_image.square() + CM_SIGMA_DATA**2).rsqrt()
    t = torch.log(sigma / CM_SIGMA_MIN) / math.log(CM_SIGMA_MAX / CM_SIGMA_MIN)

    return cskip * z + cout * model(cin * z, t, torch.zeros_like(t), labels)


@torch.no_grad()
def sample_cm(model, noise, labels, steps):
    if steps < 1:
        raise ValueError("steps must be positive")

    z = CM_SIGMA_MAX * noise
    levels = cm_noise_levels(torch.linspace(1, 0, steps + 1, device=noise.device))

    for k in range(steps):
        sigma = levels[k].expand(len(noise))
        endpoint = cm_f(model, z, sigma, labels)

        if k + 1 < steps:
            scale = (levels[k + 1].square() - CM_SIGMA_MIN**2).clamp_min(0).sqrt()
            z = endpoint + scale * torch.randn_like(noise)
        else:
            z = endpoint

    return z


cm = fresh_model()
cm_target = freeze_copy(cm)


def cm_loss(model, images, labels, step):
    grid_size = cm_grid_size(step)
    indices = torch.randint(1, grid_size + 1, (len(images),), device=DEVICE)
    sigma = cm_noise_levels(indices.float() / grid_size)
    lower = cm_noise_levels((indices.float() - 1) / grid_size)
    noise = torch.randn_like(images)
    prediction = cm_f(model, images + expand_time(sigma) * noise, sigma, labels)

    with torch.no_grad():
        target = cm_f(cm_target, images + expand_time(lower) * noise, lower, labels)

    loss = F.mse_loss(prediction, target)

    return loss, {"raw_mse": loss.detach(), "grid_intervals": grid_size}

In [ ]:
cm = train_method("CM", cm, cm_loss, sample_cm, target=cm_target)
show_training("CM")

In [ ]:
# @title CM — boundary verification / conditional generation
z, y = make_noise(NUM_CLASSES, SEED), torch.arange(NUM_CLASSES, device=DEVICE)
sigma = torch.full((len(z),), CM_SIGMA_MIN, device=DEVICE)

with torch.no_grad():
    assert torch.allclose(cm_f(cm, z, sigma, y), z, atol=1e-6)

print("CM boundary identity: PASS; CT approximation and benchmark adaptations remain")
show_conditional(cm, sample_cm, "CM", DEFAULT_NFE["CM"], RUN_INFO["CM"]["train_steps"])
cm_samples = evaluate_method("CM", sample_cm)

In [ ]:
# @title CM — 원하는 클래스 직접 생성 (라벨·seed·NFE 수정)
# 0 T-shirt, 1 Trouser, 2 Pullover, 3 Dress, 4 Coat,
# 5 Sandal, 6 Shirt, 7 Sneaker, 8 Bag, 9 Ankle boot
requested_labels = [0, 0, 1, 1, 7, 7, 9, 9]
requested_nfe = DEFAULT_NFE["CM"]
requested_seed = SEED + 20_000
custom_images = generate_conditioned(
    MODELS["CM"], sample_cm, requested_labels, requested_nfe, seed=requested_seed
)
fig, axes = plt.subplots(
    1, len(requested_labels), figsize=(2 * len(requested_labels), 2.5), squeeze=False
)

for ax, image, label in zip(axes.flat, custom_images, requested_labels):
    ax.imshow(((image[0] + 1) / 2).clamp(0, 1), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"y={label} {CLASS_NAMES[label]}", fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 4. Consistency Trajectory Models — arbitrary-time soft consistency

고정 FM teacher로 \(t\to u\)를 적분하고, EMA student로 \(u\to s\)를 이동한 결과를 online student의 \(t\to s\)와 맞춘다.
두 \(s\)-시점 예측을 frozen student로 endpoint까지 옮겨 MSE로 비교한다. **online 쪽 endpoint projection은 파라미터만 고정하고 입력 미분은 유지한다.**

\[
G_\theta(z,t,s)=z-(t-s)q_\theta(z,t,t-s,y)
=\frac{s}{t}z+(1-\frac{s}{t})g_\theta(z,t,s),\quad g_\theta=z-tq_\theta.
\]

이 residual 표현은 \(G(z,t,t)=z\)를 정확히 만족하며 zero-initialized DiT에서도 identity로 시작한다.
대각선 denoising 항 \(\|g_\theta(z_t,t,t)-x\|^2\)을 더한다.
**적응:** diffusion teacher 대신 linear FM teacher, pixel MSE, 고정 보조 손실 가중치, GAN 생략. 원 CTM의 full recipe와 구분한다.


In [ ]:
# @title CTM — soft consistency and diagonal denoising
def ctm_map(model, z, t, s, labels):
    return z - expand_time(t - s) * model(z, t, t - s, labels)


@torch.no_grad()
def sample_ctm(model, noise, labels, steps):
    if steps < 1:
        raise ValueError("steps must be positive")

    z = noise

    for k in range(steps):
        t = torch.full((len(z),), 1 - k / steps, device=z.device)
        s = torch.full_like(t, 1 - (k + 1) / steps)
        z = ctm_map(model, z, t, s, labels)

    return z


ctm = fresh_model()
ctm_target = freeze_copy(ctm)


def ctm_loss(model, images, labels, step):
    t = torch.rand(len(images), device=DEVICE)
    s = torch.rand_like(t) * t
    u = s + torch.rand_like(t) * (t - s)
    noise = torch.randn_like(images)
    z = (1 - expand_time(t)) * images + expand_time(t) * noise

    with torch.no_grad():
        at_u = integrate_fm(fm, z, t, u, labels, CTM_TEACHER_NFE)
        target_s = ctm_map(ctm_target, at_u, u, s, labels)
        target_zero = ctm_map(ctm_target, target_s, s, torch.zeros_like(s), labels)

    prediction_s = ctm_map(model, z, t, s, labels)
    # Frozen parameters, but preserve d(projection)/d(prediction_s).
    prediction_zero = ctm_map(ctm_target, prediction_s, s, torch.zeros_like(s), labels)
    consistency = F.mse_loss(prediction_zero, target_zero)
    denoised = z - expand_time(t) * model(z, t, torch.zeros_like(t), labels)
    denoise = F.mse_loss(denoised, images)

    return consistency + CTM_DENOISE_WEIGHT * denoise, {
        "soft_consistency": consistency.detach(),
        "denoise_mse": denoise.detach(),
    }

In [ ]:
ctm = train_method("CTM", ctm, ctm_loss, sample_ctm, target=ctm_target)
show_training("CTM")

In [ ]:
# @title CTM — identity verification / conditional generation
z, y = make_noise(NUM_CLASSES, SEED), torch.arange(NUM_CLASSES, device=DEVICE)
t = torch.full((len(z),), 0.6, device=DEVICE)

with torch.no_grad():
    assert torch.allclose(ctm_map(ctm, z, t, t, y), z)

assert all(p.grad is None for p in ctm_target.parameters())
print("CTM identity / frozen target: PASS")
show_conditional(
    ctm, sample_ctm, "CTM", DEFAULT_NFE["CTM"], RUN_INFO["CTM"]["train_steps"]
)
ctm_samples = evaluate_method("CTM", sample_ctm)

In [ ]:
# @title CTM — 원하는 클래스 직접 생성 (라벨·seed·NFE 수정)
# 0 T-shirt, 1 Trouser, 2 Pullover, 3 Dress, 4 Coat,
# 5 Sandal, 6 Shirt, 7 Sneaker, 8 Bag, 9 Ankle boot
requested_labels = [0, 0, 1, 1, 7, 7, 9, 9]
requested_nfe = DEFAULT_NFE["CTM"]
requested_seed = SEED + 20_000
custom_images = generate_conditioned(
    MODELS["CTM"], sample_ctm, requested_labels, requested_nfe, seed=requested_seed
)
fig, axes = plt.subplots(
    1, len(requested_labels), figsize=(2 * len(requested_labels), 2.5), squeeze=False
)

for ax, image, label in zip(axes.flat, custom_images, requested_labels):
    ax.imshow(((image[0] + 1) / 2).clamp(0, 1), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"y={label} {CLASS_NAMES[label]}", fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 5. Shortcut Models — 두 작은 이동으로 큰 이동 학습

이 절은 noise=0 → data=1 방향이다. `d=0` 표본은 FM target \(x-\epsilon\)을 학습한다.
나머지는 두 번의 \(d/2\) 이동에서 얻은 평균 속도를 stop-gradient target으로 사용해 \(d\) 이동을 학습한다.
bootstrap의 두 번째 호출은 첫 번째 예측으로 이동한 위치에서 수행하며, 두 호출 모두 같은 클래스 조건을 사용한다.
**적응:** 16단계 dyadic hierarchy, 공통 DiT와 Muon. MeanFlow의 시간 분포·adaptive weighting은 적용하지 않는다.


In [ ]:
# @title Shortcut — bootstrap objective and dyadic sampler
def shortcut_loss(model, images, labels, step):
    if SHORTCUT_LEVELS < 2 or SHORTCUT_LEVELS & (SHORTCUT_LEVELS - 1):
        raise ValueError("SHORTCUT_LEVELS must be a power of two >= 2")

    n = len(images)
    noise = torch.randn_like(images)
    t = torch.rand(n, device=DEVICE)
    mask = torch.rand(n, device=DEVICE) < SHORTCUT_FM_FRACTION
    d = torch.zeros_like(t)
    indices = (~mask).nonzero(as_tuple=True)[0]
    levels = int(math.log2(SHORTCUT_LEVELS))
    half_steps = (
        2.0 ** torch.randint(0, levels, (len(indices),), device=DEVICE)
        / SHORTCUT_LEVELS
    )
    slots = torch.round(1 / half_steps).long() - 1
    k = torch.floor(torch.rand(len(indices), device=DEVICE) * slots).long()
    t[indices] = k * half_steps
    d[indices] = 2 * half_steps
    z = (1 - expand_time(t)) * noise + expand_time(t) * images
    target = noise.neg() + images

    if len(indices):
        with torch.no_grad():
            first = model(z[indices], t[indices], half_steps, labels[indices])
            middle = z[indices] + expand_time(half_steps) * first
            second = model(middle, t[indices] + half_steps, half_steps, labels[indices])
            target[indices] = (first + second) / 2

    prediction = model(z, t, d, labels)
    loss = F.mse_loss(prediction, target)

    return loss, {
        "raw_mse": loss.detach(),
        "fm_fraction": mask.float().mean(),
        "mean_d": d.mean(),
    }


@torch.no_grad()
def sample_shortcut(model, noise, labels, steps):
    if steps < 1 or steps > SHORTCUT_LEVELS or steps & (steps - 1):
        raise ValueError("Use a power-of-two NFE within the trained hierarchy")

    z = noise

    for k in range(steps):
        t = torch.full((len(z),), k / steps, device=z.device)
        d = torch.full_like(t, 1 / steps)
        z = z + expand_time(d) * model(z, t, d, labels)

    return z

In [ ]:
shortcut = train_method("Shortcut", fresh_model(), shortcut_loss, sample_shortcut)
show_training("Shortcut")

In [ ]:
# @title Shortcut — one-step verification / conditional generation
z, y = make_noise(NUM_CLASSES, SEED), torch.arange(NUM_CLASSES, device=DEVICE)
t = torch.zeros(len(z), device=DEVICE)

with torch.no_grad():
    assert torch.allclose(
        sample_shortcut(shortcut, z, y, 1), z + shortcut(z, t, torch.ones_like(t), y)
    )

print("Shortcut forward-time one-step update: PASS")
show_conditional(
    shortcut,
    sample_shortcut,
    "Shortcut",
    DEFAULT_NFE["Shortcut"],
    RUN_INFO["Shortcut"]["train_steps"],
)
shortcut_samples = evaluate_method("Shortcut", sample_shortcut)

In [ ]:
# @title Shortcut — 원하는 클래스 직접 생성 (라벨·seed·NFE 수정)
# 0 T-shirt, 1 Trouser, 2 Pullover, 3 Dress, 4 Coat,
# 5 Sandal, 6 Shirt, 7 Sneaker, 8 Bag, 9 Ankle boot
requested_labels = [0, 0, 1, 1, 7, 7, 9, 9]
requested_nfe = DEFAULT_NFE["Shortcut"]
requested_seed = SEED + 20_000
custom_images = generate_conditioned(
    MODELS["Shortcut"],
    sample_shortcut,
    requested_labels,
    requested_nfe,
    seed=requested_seed,
)
fig, axes = plt.subplots(
    1, len(requested_labels), figsize=(2 * len(requested_labels), 2.5), squeeze=False
)

for ax, image, label in zip(axes.flat, custom_images, requested_labels):
    ax.imshow(((image[0] + 1) / 2).clamp(0, 1), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"y={label} {CLASS_NAMES[label]}", fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 6. MeanFlow — 평균 속도


\[
u_\text{target}=v_t-(t-r)\frac{d}{dt}u_\theta(z_t,r,t,y),\qquad v_t=\epsilon-x.
\]

minimal의 logit-normal pair, 75% `r=t` 표본, adaptive weighting, 현재 모델 생성 방식을 따른다.
네트워크 입력은 `(z,t,t-r,y)`이며 JVP는 `(z,t,r)` 좌표에서 tangent `(v,1,0)`을 사용한다. `t-r`은 JVP 함수 안에서 계산한다.
target 전체를 detach하고, label은 고정한다. 클래스 조건만 사용하며 classifier-free guidance는 추가하지 않는다.

adaptive loss는 `SSE/(stopgrad(SSE)+0.01)`이므로 값이 거의 1이어도 학습될 수 있다.
고정 진단 배치의 raw MSE·boundary MSE·interval MSE·cosine과 조건부 생성 그림을 함께 본다.
상한 20,000 step을 유지하고 기본적으로 5,000번째 업데이트·저장을 마친 뒤 중단한다.


In [ ]:
# @title MeanFlow — minimal objective, derivative, diagnostic, sampler
def logit_normal(n):
    return torch.sigmoid(torch.randn(n, device=DEVICE) * P_STD + P_MEAN)


def sample_meanflow_tuple(images):
    first = logit_normal(len(images))
    second = logit_normal(len(images))
    t = torch.maximum(first, second)
    r = torch.minimum(first, second)
    boundary_count = int(len(images) * DATA_PROPORTION)
    r[:boundary_count] = t[:boundary_count]
    noise = torch.randn_like(images)
    z = (1 - expand_time(t)) * images + expand_time(t) * noise

    return z, noise - images, r, t


def meanflow_outputs(model, z, velocity, r, t, labels):
    def fn(z_arg, t_arg, r_arg):
        return model(z_arg, t_arg, t_arg - r_arg, labels)

    prediction, derivative = jvp(
        fn, (z, t, r), (velocity, torch.ones_like(t), torch.zeros_like(r))
    )
    target = (velocity - expand_time(t - r) * derivative).detach()

    return prediction, target


def meanflow_loss(model, images, labels, step):
    z, velocity, r, t = sample_meanflow_tuple(images)
    prediction, target = meanflow_outputs(model, z, velocity, r, t, labels)
    sse = (prediction - target).square().flatten(1).sum(1)
    weight = (sse.detach() + NORM_EPS).reciprocal()

    return (sse * weight).mean(), {"raw_mse": sse.mean().detach() / images[0].numel()}


@torch.no_grad()
def sample_meanflow(model, noise, labels, steps):
    if steps < 1:
        raise ValueError("steps must be positive")

    z = noise

    for k in range(steps):
        t = torch.full((len(z),), 1 - k / steps, device=z.device)
        r = torch.full_like(t, 1 - (k + 1) / steps)
        z = z - expand_time(t - r) * model(z, t, t - r, labels)

    return z


# Use train data for diagnostics; final test evaluation does not choose stop steps.

with isolated_rng():
    diag_iterator = iter(train_loader)
    diag_images, diag_labels = next(diag_iterator)
    diag_images, diag_labels = (
        diag_images[:DIAG_BATCH].to(DEVICE),
        diag_labels[:DIAG_BATCH].to(DEVICE),
    )
    diag_tuple = sample_meanflow_tuple(diag_images)


@torch.no_grad()
def meanflow_diagnostic(model):
    z, velocity, r, t = diag_tuple
    prediction, target = meanflow_outputs(model, z, velocity, r, t, diag_labels)
    mse = (prediction - target).square().flatten(1).mean(1)
    cosine = F.cosine_similarity(prediction.flatten(1), target.flatten(1), dim=1)
    boundary = r == t
    interval = r < t

    return {
        "boundary_mse": float(mse[boundary].mean()),
        "interval_mse": float(mse[interval].mean()),
        "interval_cosine": float(cosine[interval].mean()),
    }

In [ ]:
meanflow = train_method(
    "MeanFlow",
    fresh_model(),
    meanflow_loss,
    sample_meanflow,
    diagnostic=meanflow_diagnostic,
)
show_training("MeanFlow")

In [ ]:
# @title MeanFlow — JVP chain rule / boundary / one-step generation
z, y = make_noise(NUM_CLASSES, SEED), torch.arange(NUM_CLASSES, device=DEVICE)
t = torch.full((len(z),), 0.6, device=DEVICE)
velocity = torch.randn_like(z)
_, boundary_target = meanflow_outputs(meanflow, z, velocity, t, t, y)
assert torch.allclose(boundary_target, velocity)


def analytic_fn(zz, tt, rr):
    return zz + expand_time(tt) + 2 * expand_time(tt - rr)


_, analytic_derivative = jvp(
    analytic_fn, (z, t, t / 2), (velocity, torch.ones_like(t), torch.zeros_like(t))
)
assert torch.allclose(analytic_derivative, velocity + 3)

with torch.no_grad():
    one = torch.ones_like(t)
    assert torch.allclose(
        sample_meanflow(meanflow, z, y, 1), z - meanflow(z, one, one, y)
    )

print("MeanFlow chain rule / r=t target / one-step update: PASS")
show_conditional(
    meanflow,
    sample_meanflow,
    "MeanFlow",
    DEFAULT_NFE["MeanFlow"],
    RUN_INFO["MeanFlow"]["train_steps"],
)
meanflow_samples = evaluate_method("MeanFlow", sample_meanflow)

In [ ]:
# @title MeanFlow — 원하는 클래스 직접 생성 (라벨·seed·NFE 수정)
# 0 T-shirt, 1 Trouser, 2 Pullover, 3 Dress, 4 Coat,
# 5 Sandal, 6 Shirt, 7 Sneaker, 8 Bag, 9 Ankle boot
requested_labels = [0, 0, 1, 1, 7, 7, 9, 9]
requested_nfe = DEFAULT_NFE["MeanFlow"]
requested_seed = SEED + 20_000
custom_images = generate_conditioned(
    MODELS["MeanFlow"],
    sample_meanflow,
    requested_labels,
    requested_nfe,
    seed=requested_seed,
)
fig, axes = plt.subplots(
    1, len(requested_labels), figsize=(2 * len(requested_labels), 2.5), squeeze=False
)

for ax, image, label in zip(axes.flat, custom_images, requested_labels):
    ax.imshow(((image[0] + 1) / 2).clamp(0, 1), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"y={label} {CLASS_NAMES[label]}", fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 7. 질문별 비교

공통 표에서는 생성 품질·클래스 일치율·실제 학습 step·시간을 본다. 조기 종료 시점이 다르면 동일 학습 budget의 우열 비교로 해석하지 않는다.
RF2의 총 비용에는 RF1, CTM에는 FM teacher 학습 비용을 더한다. `train_s`에는 해당 절의 teacher 추론·진단·주기적 그림 저장이 포함된다.
CM과 CTM의 EMA target은 학습 목표의 구성 요소다. 최종 생성은 모든 방법에서 마지막 online 가중치를 사용한다.
클래스 조건부 생성은 guidance 추가 호출이 없어 현재 sampler에서는 step 수와 NFE가 같다.


In [ ]:
# @title Compare A — final quality and actual training cost
summary = pd.DataFrame(RESULTS).T
summary["total_train_s"] = summary["train_s"]
summary.loc["RF2", "total_train_s"] += RUN_INFO["RF1"]["train_s"]
summary.loc["CTM", "total_train_s"] += RUN_INFO["FM"]["train_s"]
summary["dependency"] = ""
summary.loc["RF2", "dependency"] = "RF1"
summary.loc["CTM", "dependency"] = "FM"
display(
    summary[
        [
            "train_steps",
            "stop_reason",
            "nfe",
            "feature_fid",
            "kid",
            "condition_accuracy",
            "train_s",
            "total_train_s",
            "dependency",
            "optimizer_backend",
        ]
    ]
)
summary.to_csv(RUN_DIR / "summary.csv")

SAMPLERS = {
    "FM": sample_fm,
    "RF1": sample_fm,
    "RF2": sample_fm,
    "CM": sample_cm,
    "CTM": sample_ctm,
    "Shortcut": sample_shortcut,
    "MeanFlow": sample_meanflow,
}

In [ ]:
# @title Compare B — matched labels/noise, NFE-quality curves
budget_rows = []
eval_labels = balanced_labels(EVAL_N)

for name, sampler in SAMPLERS.items():
    for nfe in NFE_LIST:
        fake = generate_conditioned(MODELS[name], sampler, eval_labels, nfe)
        row = {"method": name, "nfe": nfe, **quality_metrics(fake, eval_labels)}
        budget_rows.append(row)
        print(row)

budget_table = pd.DataFrame(budget_rows)
budget_table.to_csv(RUN_DIR / "nfe_quality.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for name, group in budget_table.groupby("method"):
    axes[0].plot(group.nfe, group.feature_fid, marker="o", label=name)
    axes[1].plot(group.nfe, group.condition_accuracy, marker="o", label=name)

for ax, ylabel in zip(axes, ("feature-FID (lower)", "condition accuracy (higher)")):
    ax.set_xscale("log", base=2)
    ax.set_xlabel("NFE")
    ax.set_ylabel(ylabel)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# @title Compare C — FM vs RF trajectory geometry
def path_geometry(trajectory):
    flat = trajectory.float().flatten(2)
    increments = flat[1:] - flat[:-1]
    length = increments.norm(dim=-1).sum(0)
    chord = (flat[-1] - flat[0]).norm(dim=-1).clamp_min(1e-8)
    curvature = (flat[2:] - 2 * flat[1:-1] + flat[:-2]).norm(dim=-1).mean()

    return {
        "path_length_ratio": float((length / chord).mean()),
        "discrete_curvature": float(curvature),
    }


geometry_labels = balanced_labels(NUM_CLASSES * SAMPLES_PER_CLASS)
geometry_noise = make_noise(len(geometry_labels), SEED + 10_000)
geometry_rows = []

for name in ("FM", "RF1", "RF2"):
    trajectories = []

    for start in range(0, len(geometry_labels), SAMPLE_BATCH):
        noise = geometry_noise[start : start + SAMPLE_BATCH]
        labels = geometry_labels[start : start + SAMPLE_BATCH]
        t = torch.ones(len(noise), device=DEVICE)
        _, trajectory = integrate_fm(
            MODELS[name],
            noise,
            t,
            torch.zeros_like(t),
            labels,
            RF_TEACHER_NFE,
            return_traj=True,
        )
        trajectories.append(trajectory)

    geometry_rows.append(
        {"method": name, **path_geometry(torch.cat(trajectories, dim=1))}
    )

display(pd.DataFrame(geometry_rows))
print(
    "Reflow 이후 경로가 더 곧아졌는지는 측정 결과로 판단합니다. 품질 향상과 동일한 조건은 아닙니다."
)

### CM과 CTM에서 확인할 차이

CM은 하나의 noise level에서 near-data endpoint를 반환한다. 반복 생성은 예측한 endpoint에 다시 noise를 더해 수행한다.
CTM은 시작·도착시간을 받아 중간 위치를 반환하므로, 같은 라벨에서 `t→s→r`과 `t→r`의 composition을 비교할 수 있다.
CM의 additive noise와 CTM의 linear FM teacher는 서로 다른 경로다. 아래 CM 지표는 인접 CT 입력의 차이, CTM 지표는 map composition 오차로 각각 해석한다.


In [ ]:
# @title Compare D — CM adjacent consistency / CTM composition
with torch.no_grad(), isolated_rng():
    images, labels = next(iter(test_loader))
    images, labels = images[:SAMPLE_BATCH].to(DEVICE), labels[:SAMPLE_BATCH].to(DEVICE)
    noise = torch.randn_like(images)
    grid = cm_grid_size(max(1, RUN_INFO["CM"]["train_steps"]))
    i = torch.randint(1, grid + 1, (len(images),), device=DEVICE)
    sigma, lower = (
        cm_noise_levels(i.float() / grid),
        cm_noise_levels((i.float() - 1) / grid),
    )
    a = cm_f(cm, images + expand_time(sigma) * noise, sigma, labels)
    b = cm_f(cm, images + expand_time(lower) * noise, lower, labels)
    cm_adjacent_mse = float(F.mse_loss(a, b))
    t, s, r = (
        torch.full((len(images),), value, device=DEVICE) for value in (1.0, 0.5, 0.0)
    )
    direct = ctm_map(ctm, noise, t, r, labels)
    composed = ctm_map(ctm, ctm_map(ctm, noise, t, s, labels), s, r, labels)
    print(
        {
            "cm_adjacent_ct_mse": cm_adjacent_mse,
            "ctm_composition_mse": float(F.mse_loss(direct, composed)),
        }
    )

## 8. Flow Map 관점 — 조건부 finite-time map

공통 입력 공간은 \(\mathbb R^{1\times32\times32}\), 클래스 \(y\)는 모든 이동 동안 고정한다.
FM/RF는 ODE 적분, CTM은 trajectory map, Shortcut은 전진 구간 속도, MeanFlow는 역방향 평균 속도로 이동을 표현한다.
CM은 noise scale에서 \(\sigma_\min\) endpoint로 가는 특수한 map이다. 임의 구간 이동으로 확장해 해석하지 않는다.

GFM 논문의 Euclidean 관점으로 기존 함수를 재표현한다. 이 절에서는 새 모델을 학습하지 않는다.
representation 등식은 구현 검증이고, composition 오차는 학습 결과의 진단이다. MeanFlow composition은 직접 학습한 손실이 아니다.


In [ ]:
# @title Flow Map — adapters and identity checks with labels
@torch.no_grad()
def flow_map(name, model, z, t, s, labels, nfe=16):
    if name in ("FM", "RF1", "RF2"):
        return integrate_fm(model, z, t, s, labels, nfe)

    if name == "CTM":
        return ctm_map(model, z, t, s, labels)

    if name == "MeanFlow":
        return z - expand_time(t - s) * model(z, t, t - s, labels)

    if name == "Shortcut":
        return z + expand_time(s - t) * model(z, t, s - t, labels)

    raise ValueError("CM uses its own noise-scale endpoint adapter")


@torch.no_grad()
def cm_endpoint_map(model, z, sigma, labels):
    return cm_f(model, z, sigma, labels)


z, labels = make_noise(NUM_CLASSES, SEED), torch.arange(NUM_CLASSES, device=DEVICE)
one = torch.ones(len(z), device=DEVICE)
zero = torch.zeros_like(one)

with torch.no_grad():
    for name in ("FM", "RF1", "RF2", "CTM", "MeanFlow"):
        nfe = DEFAULT_NFE[name] if name in ("FM", "RF1", "RF2") else 1
        mapped = flow_map(name, MODELS[name], z, one, zero, labels, nfe=nfe)
        sampled = SAMPLERS[name](MODELS[name], z, labels, nfe)
        assert torch.allclose(mapped, sampled, atol=1e-6)

    assert torch.allclose(
        flow_map("Shortcut", shortcut, z, zero, one, labels),
        sample_shortcut(shortcut, z, labels, 1),
        atol=1e-6,
    )
    sigma = torch.full_like(one, CM_SIGMA_MAX)
    assert torch.allclose(
        cm_endpoint_map(cm, CM_SIGMA_MAX * z, sigma, labels),
        sample_cm(cm, z, labels, 1),
        atol=1e-5,
        rtol=1e-5,
    )

print("Conditional flow-map adapters equal sampler updates: PASS")

composition_rows = []

for name in ("CTM", "Shortcut", "MeanFlow"):
    t, r = (zero, one) if name == "Shortcut" else (one, zero)
    s = (t + r) / 2
    direct = flow_map(name, MODELS[name], z, t, r, labels)
    middle = flow_map(name, MODELS[name], z, t, s, labels)
    composed = flow_map(name, MODELS[name], middle, s, r, labels)
    composition_rows.append(
        {"method": name, "composition_mse": float(F.mse_loss(direct, composed))}
    )

display(pd.DataFrame(composition_rows))

### 해석 순서

1. 실행 검증이 통과했는지 확인한다. 통과는 학습 수렴이나 생성 품질을 보장하지 않는다.
2. 각 모델의 클래스별 그림에서 요청한 의류 형태가 나오는지 확인한다.
3. raw error와 조건 일치율, NFE-quality 곡선을 함께 본다.
4. 실제 학습 step, 중단 이유, teacher 비용을 확인한 뒤 모델 간 결과를 비교한다.
5. 모든 모델의 기본 중단 시점은 각각 5,000 step이다. RF1과 RF2는 각 단계에서 5,000 step씩 학습한다. 모델별 조정은 `EARLY_STOP_AT`에서 지정한다.

각 모델 폴더에는 조건부 생성 PNG, `training.csv`, `last.pt`가 저장된다. 체크포인트는 현재 모델·optimizer·CM/CTM target·설정·중단 step을 담는다.
데이터 iterator 위치를 복원하는 완전 동일한 재개 기능은 포함하지 않는다. 새 실습을 재실행하면 별도의 run 폴더를 만든다.
